<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Step 1: Data Contract — Unit of Analysis & Time Window

#### Contract Statement (5 Answers):
1. **Grain (One Row):** $1\text{ Row} = 1\text{ Pseudonymized Web Page (Content Item)}$ represented by a unique `content_hash_id` aggregated over a monthly decision window $T_0$.
2. **Table(s) Used:** Primary daily performance table `fact_content_daily_performance` joined with `dim_content` metadata via `content_hash_id`.
3. **Time Window:** Mid-Panel Month **`2026-03`** (March 1, 2026 to March 31, 2026).
4. **Target / Label:** Binary traffic decay proxy (`gsc_clicks < 30`) and continuous Decay Risk Score for page refresh prioritization.
5. **Deliberate Exclusion:** Future outcome data (Months 4–6, i.e., April–June 2026 logs) to strictly avoid Data Leakage.

#### Verification Findings:
* **Raw Daily Rows:** ~9.84 Million daily log records in March 2026.
* **Unique Content Items:** Exactly **331,437 unique web pages**.
* **Date Span:** Full calendar month from `2026-03-01` to `2026-03-31`.

In [8]:
import duckdb
from huggingface_hub import get_token

# 1. Token Setup
token = get_token()
# Agar token None return kare, toh neeche wali line se comment (#) hatayein:
# token = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# 2. Connection Setup
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 3. Verified Step 1 Query
verify_query = f"""
SELECT
    COUNT(*) as total_daily_raw_rows,
    COUNT(DISTINCT content_hash_id) as unique_content_rows_grain,
    MIN(report_date) as slice_start_date,
    MAX(report_date) as slice_end_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

# Execute and Display Result
df_verification = con.sql(verify_query).df()

print("=" * 60)
print("VERIFICATION RESULT: UNIT OF ANALYSIS & TIME WINDOW")
print("=" * 60)
print(df_verification)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

VERIFICATION RESULT: UNIT OF ANALYSIS & TIME WINDOW
   total_daily_raw_rows  unique_content_rows_grain slice_start_date  \
0               9841378                     331437       2026-03-01   

  slice_end_date  
0     2026-03-31  


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Step 2: Field Sorting (Feature / Label / Context / Excluded)

Below is the classification of all warehouse fields into the four required buckets for **Lane 2: Content Refresh & Decay Predictor** during decision moment $T_0$ (`month = '2026-03'`):

| Bucket | Field Name | Description & Role |
| :--- | :--- | :--- |
| **1. Feature** | `gsc_clicks` (Aggregated) | Total organic clicks generated in the 30-day window ($T_0$). |
| **1. Feature** | `gsc_impressions` (Aggregated) | Total search impressions received in the 30-day window ($T_0$). |
| **1. Feature** | `gsc_avg_position` (Aggregated) | Average Google SERP ranking position recorded prior to $T_0$. |
| **1. Feature** | `ga4_total_engagement_sec` | Total user engagement duration (seconds) logged before $T_0$. |
| **1. Feature** | `sessions_organic` (Aggregated) | Count of organic traffic sessions logged up to cutoff $T_0$. |
| **2. Label** | `target_decay_flag` | Binary target derivative (`1` if future traffic drops below baseline, `0` otherwise). |
| **2. Label** | `decay_risk_score` | Continuous proxy score calculated to rank pages urgently requiring content refresh. |
| **3. Context** | `content_hash_id` | Primary Key / Grain identifier (Pseudonymized Content Item). |
| **3. Context** | `client_hash_id` | Foreign Key / Grouping identifier (Pseudonymized Client Account). |
| **3. Context** | `report_date` / `month` | Temporal metadata defining the 30-day panel window ($T_0 = \text{'2026-03'}$). |
| **4. Excluded** | `month = '2026-04'` to `'2026-06'` | **WHY:** Post-cutoff performance data. Including future metrics creates severe **Data Leakage**, causing artificial $100\%$ accuracy during training. |
| **4. Excluded** | `ai_chatgpt`, `ai_perplexity`, etc. | **WHY:** Sparse AI-referral metrics in this panel; excluded to maintain a high signal-to-noise ratio and avoid zero-variance noise. |
| **4. Excluded** | `client_has_gsc`, `client_has_ga4` | **WHY:** Static boolean flags that do not vary per page; filtered out during the data availability check step (`IS TRUE`). |

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.